# 6.2 Karakters tellen — de baseline vóór embeddings

6.1 drukte *kolommen* plat naar twee dimensies. Dit notebook gaat over de stap daarvóór: hoe wordt een stukje *tekst* überhaupt een rij getallen? Het goedkoopste antwoord is tellen — welke sequenties van drie karakters een tekstblok bevat — en die telling de vingerafdruk noemen. `CountVectorizer` is een opzoektabel, geen geleerde functie: jij kiest de granulariteit, hij telt. Geen GPU, geen download, binnen enkele seconden gefit op elke machine.

Het is ook het getal dat verslagen moet worden. [06.3](06.3-vectorspaces%20jeffrey.ipynb) vervangt deze handmatig ontworpen vector door een vector die een neuraal netwerk heeft geleerd van miljoenen documenten, en [06.4](06.4-your-own-vectors%20jeffrey.ipynb) laat de twee tegen elkaar racen op dezelfde taak; een embedding verdient zijn plek pas als hij beter presteert dan het tellen van trigrammen.

De showcase is een echte gepubliceerde claim. In 2020 analyseerde het Zwitserse bedrijf OrphAnalytics elk bericht dat werd geplaatst door "Q" — het anonieme account achter QAnon — en concludeerde uit de schrijfstijl dat de berichten waren geschreven door **twee verschillende personen**: één op 4chan eind 2017, en een tweede die het overnam na de verhuizing naar 8chan ([white paper](https://www.orphanalytics.com/en/news/whitepaper202012/OrphAnalyticsQAnon2020.pdf), [persbericht](https://www.prnewswire.com/news-releases/qanon-is-two-different-people-shows-machine-learning-analysis-from-orphanalytics-301192981.html)). Hun methode, ontdaan van de "gepatenteerde technologie": knip de tekst in gelijke stukken, tel karakter-trigrammen, meet hoe ver de stukken uit elkaar liggen, bekijk het plaatje. Vier stappen, en elk daarvan is een beslissing. Dit notebook bouwt het na in een paar regels `sklearn`, en stelt vervolgens de vraag die les 5 ons leerde: wat ondersteunt het plaatje nu werkelijk?

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from goad_toolkit.config import FileConfig
from goad_toolkit.datatransforms import Filter, Pipeline, TransformBase
from goad_toolkit.filehandler import FileHandler
from goad_toolkit.visualizer import PlotSettings, ProjectionPlot
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import manhattan_distances

from wa_analyzer.data import load_own_chat
from wa_analyzer.model import TextClustering


## 6.2.1 De data: elk bericht, drie platforms

Alle berichten van Q, verzameld door een vrijwilligersproject op GitHub als één JSON-bestand. `FileHandler` downloadt het eenmalig naar `~/.cache/mads-dav/raw` — dezelfde caching die 03.2 gebruikte voor de zonnevlekken, zodat er niets tweemaal wordt opgehaald. Het bestand is JSON in plaats van CSV, dus de handler haalt het alleen op; `json_normalize` vlakt de geneste metadata van berichten af tot kolommen.

In [ ]:
config = FileConfig(
    data_dir=Path.home() / ".cache/mads-dav",
    filename=Path("qanon-posts.json"),
    url="https://raw.githubusercontent.com/jkingsman/JSON-QAnon/main/posts.json",
    date_column=None,
)
FileHandler(config).download()

# encoding spelled out: Windows would otherwise pick a locale-specific default
with (config.data_dir / "raw" / config.filename).open(encoding="utf-8") as f:
    posts = pd.json_normalize(json.load(f)["posts"], sep="_")

posts["time"] = pd.to_datetime(posts["post_metadata_time"], unit="s")
posts["site"] = posts["post_metadata_source_site"]
posts = posts.sort_values("time").reset_index(drop=True)

print(f"{len(posts):,} posts, median {posts.text.fillna('').str.len().median():.0f} characters each")
posts.groupby("site").time.agg(["min", "max", "size"])


Eén rij is één bericht, en een bericht is kort: de helft is korter dan honderd tekens — een "Q"-handtekening, een link, een antwoord van één regel. De drie platforms zijn geen drie parallelle bronnen, maar drie **periodes**: 4chan gedurende de eerste twee maanden, 8chan totdat het in 2019 offline ging, en daarna 8kun. Houd die tabel in gedachten. Elk label in het plaatje van dit notebook is een platform, en een platform is ook een datum.

## 6.2.2 Schonen in stappen, en wat één rij wordt

Twee beslissingen vóór het tellen, elk een pipelinestap zodat deze geprint en bediscussieerd kan worden. `NormaliseText` — hier geschreven, met dezelfde structuur als de stappen uit les 1 — zet alles om naar kleine letters en voegt witruimte samen, en doet in deze eerste ronde niets meer: het telt wat er geplaatst is, inclusief links, en §6.2.5 vraagt zich af of dat verstandig was. `Filter` verwijdert vervolgens berichten korter dan vijftig tekens, omdat een eenzame "Q" of een kale link geen schrijfstijlproef is.

In [ ]:
class NormaliseText(TransformBase):
    """Lowercase and collapse whitespace; optionally strip URLs, or keep only letters and spaces.

    Also records the cleaned length as `n_chars`, which the chunking below works from.
    """

    def transform(self, data: pd.DataFrame, column: str, feature: str = "text",
                  strip_urls: bool = False, letters_only: bool = False) -> pd.DataFrame:
        text = data[column].fillna("").astype(str).str.lower()
        if strip_urls:
            text = text.str.replace(r"https?://\S+", " ", regex=True)
        if letters_only:
            text = text.str.replace(r"[^a-z ]+", " ", regex=True)
        text = text.str.replace(r"\s+", " ", regex=True).str.strip()
        data[feature] = text
        data["n_chars"] = text.str.len()
        return data


cleaning = Pipeline().add(NormaliseText, column="text").add(Filter, expr="n_chars > 50")
clean = cleaning.apply(posts)
print(cleaning)
print(f"{len(clean):,} posts kept, {clean.n_chars.sum():,} characters in total")


### Chunks: de rij die de methode nodig heeft

De eenheid van het artikel is niet het bericht, maar een chunk (tekstblok) van ongeveer 7.500 tekens, en de reden is rekenkundig. Een trigram-telling op een bericht van honderd tekens bestaat bijna alleen uit nullen, en twee korte berichten verschillen vooral door wat er toevallig in stond; een telling over duizenden tekens stabiliseert in frequenties die echt over de auteur gaan. De chunks zijn bovendien allemaal **even groot**, en dat maakt zometeen uit voor de afstand: een langere chunk heeft grotere tellingen, waardoor ongelijke chunks ver uit elkaar zouden lijken te liggen om geen enkele andere reden dan hun lengte.

`ChunkByCharacters` nummert opeenvolgende berichten zodat elke chunk ongeveer hetzelfde aantal tekens bevat — honderd chunks van alles wat de opschoning overleefde — en een `groupby` voegt de tekst samen en behoudt het platform waar de meeste berichten van de chunk vandaan kwamen.

In [ ]:
class ChunkByCharacters(TransformBase):
    """Number consecutive rows so that every chunk holds about `size` characters of `column`.

    With `by`, the numbering restarts for every group — one run of chunks per author, say.
    """

    def transform(self, data: pd.DataFrame, column: str, size: int, feature: str = "chunk",
                  by: str | None = None) -> pd.DataFrame:
        lengths = data[column].str.len()
        running = lengths.groupby(data[by]).cumsum() if by else lengths.cumsum()
        data[feature] = running // size
        return data


def build_chunks(frame: pd.DataFrame, size: int, label: str, by: str | None = None) -> pd.DataFrame:
    """Join consecutive rows into chunks of about `size` characters, each labelled with the
    `label` value most of its rows carry. The remainder at the end is dropped."""
    keys = [by, "chunk"] if by else ["chunk"]
    numbered = Pipeline().add(ChunkByCharacters, column="text", size=size, by=by).apply(frame)
    chunks = numbered.groupby(keys).agg(
        text=("text", " ".join),
        label=(label, lambda s: s.mode().iloc[0]),
        n_posts=("text", "size"),
    ).reset_index()
    return chunks[chunks.text.str.len() > size / 2].reset_index(drop=True)


N_CHUNKS = 100
chunk_size = clean.n_chars.sum() // N_CHUNKS
chunks = build_chunks(clean, size=chunk_size, label="site")

print(f"{len(chunks)} chunks of about {chunk_size:,} characters, {chunks.n_posts.median():.0f} posts each")
print(chunks.label.value_counts().to_string())


## 6.2.3 Van tekst naar getallen: karakter-trigrammen

Een trigram is een reeks van drie opeenvolgende tekens, spaties inbegrepen. `CountVectorizer(analyzer="char", ngram_range=(3, 3))` vindt elk trigram dat ergens in het corpus voorkomt, maakt er een kolom van, en telt hoe vaak elke tekst het bevat. Twee speelgoed-strings laten het hele mechanisme zien — "yellow banana" en "papagena papaya banana" — en de kolommen die de moeite waard zijn om te bekijken zijn degene die vaker dan één keer voorkomen:

In [ ]:
example = ["yellow banana", "papagena papaya banana"]
toy = CountVectorizer(analyzer="char", ngram_range=(3, 3))
toy_counts = toy.fit_transform(example).toarray()

table = pd.DataFrame(toy_counts, index=pd.Index(example), columns=toy.get_feature_names_out())
print(f"{table.shape[1]} distinct trigrams, for example {list(table.columns[:6])}")
table.loc[:, table.sum() > 1]


Dertien en drieëntwintig tekens werden twee rijen van 23 tellingen. `ana` is de kolom die beide strings delen — elk twee keer, omdat "banana" het twee keer bevat — en `apa` en `pap` zijn de tweede string die zichzelf herhaalt in "papagena papaya". Het grootste deel van elke rij is nul, aangezien elk trigram in het corpus een kolom is en elke afzonderlijke tekst er maar weinig bevat; dat blijft op schaal gelden, waar de honderd chunks hieronder zo'n twintigduizend unieke trigrammen beslaan.

### Manhattan-afstand: hoe verschillend zijn twee rijen met tellingen?

Twee rijen, één getal voor hoe ver ze uit elkaar liggen. De **Manhattan-afstand** telt het absolute verschil per kolom bij elkaar op — `|2 − 2| + |1 − 0| + …` — waarbij je blok voor blok door het raster wandelt in plaats van diagonaal af te snijden zoals de Euclidische afstand zou doen. Op tellingen leest het direct af: *deze twee teksten verschillen in totaal met 21 trigram-voorkomens*. Identieke rijen hebben afstand nul. Dit is ook waarom de chunks even groot moesten zijn: een chunk die twee keer zo lang is heeft grofweg twee keer zo grote tellingen, en Manhattan zou hem alleen al om zijn lengte ver van alles vandaan plaatsen. [06.3](06.3-vectorspaces%20jeffrey.ipynb) introduceert cosinus-similariteit, die per definitie lengte negeert; hier doen gelijke chunks hetzelfde werk.

In [ ]:
manhattan_distances(toy_counts, toy_counts)


Nu het echte werk: honderd chunks, elk trigram in het corpus als kolom, en de matrix van honderd bij honderd afstanden ertussen.

In [ ]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3))
counts = vectorizer.fit_transform(chunks.text).toarray()
distance = manhattan_distances(counts, counts)

off_diagonal = distance[np.triu_indices(len(distance), k=1)]
print(f"counts: {counts.shape[0]} chunks x {counts.shape[1]:,} trigrams; "
      f"a chunk uses {np.mean(counts > 0):.0%} of them")
print(f"distance: {distance.shape}; nearest pair {off_diagonal.min():,.0f} apart, "
      f"farthest {off_diagonal.max():,.0f}")


## 6.2.4 Reduceer, en kijk

Elke rij van de afstandsmatrix beschrijft één chunk aan de hand van zijn afstand tot alle honderd andere — een honderddimensionaal profiel. Twee chunks met vergelijkbare profielen lijken op elkaar, en PCA op de profielen is de platte schaduw uit 6.1, met twee componenten ervan. Kleur is het platform, wat tevens de periode is.

In [ ]:
coords = PCA(n_components=2).fit_transform(distance)

settings = PlotSettings(
    figsize=(7, 6),
    title="A hundred chunks of Q by trigram counts: the 4chan months sit apart",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=coords, labels=chunks.label.to_numpy(), palette="Set1")


De tien chunks uit de 4chan-maanden liggen los van de rest; de 8chan- en 8kun-chunks liggen bovenop elkaar. Dat is min of meer het plaatje uit het artikel — en de regel uit 6.1 geldt voordat we het geloven: een projectie vult altijd zijn kader, dus haal het getal op dat niet van het plaatje afhangt. De silhouette per platform, berekend op de volledige afstandsmatrix, afgezet tegen wat dezelfde score oplevert wanneer de platformlabels worden geschud:

In [ ]:
def separation(distance: np.ndarray, labels: np.ndarray, seed: int = 0) -> tuple[float, float]:
    '''Silhouette of the labels on a precomputed distance matrix, and the same for shuffled labels.'''
    rng = np.random.default_rng(seed)
    real = silhouette_score(distance, labels, metric="precomputed")
    shuffled = np.mean([silhouette_score(distance, rng.permutation(labels), metric="precomputed")
                        for _ in range(50)])
    return float(real), float(shuffled)


labels = chunks.label.to_numpy()
real, shuffled = separation(distance, labels)
first_months = np.where(labels == "4ch", "4chan", "after the move")
early_real, early_shuffled = separation(distance, first_months)

print(f"silhouette by site:          {real:.3f}   (shuffled labels: {shuffled:.3f})")
print(f"silhouette 4chan vs. after:  {early_real:.3f}   (shuffled labels: {early_shuffled:.3f})")


Echt, en klein. Geschudde labels scoren ongeveer nul, zoals het hoort; de echte platforms scoren een paar honderdsten daarboven, en vrijwel dat hele verschil komt door de tien 4chan-chunks tegenover al het andere — de splitsing tussen 8chan en 8kun voegt niets toe. De bevinding die de tellingen tot nu toe ondersteunen is dus smaller dan "twee personen": **de eerste twee maanden lezen anders dan de rest.**

### Welke trigrammen zorgen voor de scheiding

Een telvector kan worden teruggelezen, wat het grote voordeel is van een handmatig ontworpen representatie boven een geleerde representatie. Per trigram: het aandeel van een chunk op 4chan minus het aandeel na de verhuizing; de uitersten zijn het verschil in een lijstje:

In [ ]:
share = counts / counts.sum(axis=1, keepdims=True)
early = labels == "4ch"
difference = pd.Series(share[early].mean(axis=0) - share[~early].mean(axis=0),
                       index=vectorizer.get_feature_names_out())

print("more common in the 4chan months:", [repr(t) for t in difference.nlargest(10).index])
print("more common after the move:     ", [repr(t) for t in difference.nsmallest(10).index])


Twee totaal verschillende soorten dingen staan op die lijsten. De 4chan-kant bestaat uit **vragen** — `wha`, `why`, `? w`, `is ` — wat een feit is over het schrijven: de vroege berichten waren reeksen Socratische vragen. De andere kant bestaat uit niets anders dan **links**: `htt`, `://`, `com`, `om/`. Een link wordt geplakt, niet getypt, en het domein erin duidt op het platform in plaats van de schrijver. Een persoon die van 4chan naar 8chan verhuist en begint te antwoorden met links verandert zijn trigramprofiel zonder te veranderen wie hij is — en een trigramteller kan het verschil tussen die twee soorten verandering niet zien.

## 6.2.5 Verifieer vóórdat je een mechanisme claimt

De regel uit les 3: controleer het mechanisme dat je op het punt staat op te schrijven. De goedkoopste alternatieve verklaring voor het plaatje is "de markup van het platform", en die kan worden verwijderd: `NormaliseText` neemt nog twee beslissingen — verwijder URL's, behoud alleen letters en spaties — dus de tweede doorgang is dezelfde pipeline met twee aangepaste parameters en al het andere identiek. De tel- en afstandsstappen zijn de stappen die `wa_analyzer.model.TextClustering` inpakt, dus vanaf hier is het één functieaanroep:

In [ ]:
clustering = TextClustering()
assert np.allclose(clustering.fit(chunks.text.tolist()), distance)  # noqa: S101 -- the class is the cells above

stripped = (
    Pipeline()
    .add(NormaliseText, column="text", strip_urls=True, letters_only=True)
    .add(Filter, expr="n_chars > 50")
    .apply(posts)
)
letter_chunks = build_chunks(stripped, size=stripped.n_chars.sum() // N_CHUNKS, label="site")
letter_distance = clustering.fit(letter_chunks.text.tolist())
letter_labels = letter_chunks.label.to_numpy()
letter_early, _ = separation(letter_distance, np.where(letter_labels == "4ch", "4chan", "after the move"))

print(f"silhouette, 4chan vs. after:  everything counted {early_real:.3f}   "
      f"letters and spaces only {letter_early:.3f}")

passes = PlotSettings(
    figsize=(12, 5),
    title="The same posts, counted twice",
    subplot_titles=["everything, links and markup included", "letters and spaces only"],
    xlabel="",
    ylabel="",
    max_cols=2,
)
host = ProjectionPlot(passes)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(ProjectionPlot(passes), axes[0], coordinates=coords, labels=labels,
                  palette="Set1", legend=False)
host.plot_on_axes(ProjectionPlot(passes), axes[1], coordinates=clustering.reduce_dims(letter_distance),
                  labels=letter_labels, palette="Set1")


De 4chan-chunks liggen net zo duidelijk apart zodra elke link, cijfer en leesteken verdwenen is — het silhouette verschuift niet. De scheiding zit dus niet in de markup van het platform: het zit in de woorden. Dat sluit de goedkoopste verklaring uit, en laat de twee over die de tellingen niet uit elkaar kunnen halen. Leg het langs het raster van 05.1:

- **Bewijs (Evidence).** Een reële structuur, klein, steunend op tien chunks tegenover negentig, en het patroon is "de eerste twee maanden verschillen", niet "twee clusters".
- **Mechanisme (Mechanism).** Twee kandidaten, en beide voorspellen exact dit plaatje: een tweede auteur nam het over, óf dezelfde auteur veranderde van register — van Socratische vragen op het ene forum naar antwoorden en links op het andere. De wisseling van auteur, als die er al was, valt per definitie samen met de verhuizing van platform, dus niets in dit corpus kan ze uit elkaar trekken.
- **Replicatie (Replication).** Niet beschikbaar. Eén corpus, één overstap.

**Oordeel: aannemelijk, onbewezen.** Wat de doorslag zou geven is tekst van dezelfde bekende schrijver over eenzelfde soort platformverhuizing, en niemand heeft dat. Het persbericht zei "twee verschillende personen"; de telvectoren zeggen "een verandering in de schrijfstijl, op het moment dat het platform veranderde, en niet veroorzaakt door platform-markup". Die tweede zin is degene die dit notebook kan verdedigen.

## 6.2.6 Jouw beurt

Dezelfde vier stappen, je eigen chat, met één verschil dat je al weet te hanteren: de auteurs zijn gelabeld, dus de vraag is niet "hoeveel personen" maar **scheidt de typstijl de mensen waarvan je weet dat ze er zijn?** Per auteur: normaliseren, chunken, tellen, afstand — `ChunkByCharacters(by="author")` herstart de chunk-nummering per persoon — daarna het plaatje en het getal.

De chunkgrootte is opnieuw een beslissing. Een chat heeft veel minder tekst per persoon dan Q plaatste, dus de chunks hieronder zijn 500 tekens in plaats van 8.000, en de tellingen daarin bevatten meer ruis. Verhoog het als je voldoende tekst hebt, en kijk wat het silhouette doet; verlaag het en zie het dalen.

In [ ]:
own = load_own_chat().sort_values(["author", "timestamp"])

own_clean = (
    Pipeline()
    .add(NormaliseText, column="message", strip_urls=True)
    .add(Filter, expr="n_chars > 20")
    .apply(own)
)

OWN_CHUNK = 500
own_chunks = build_chunks(own_clean, size=OWN_CHUNK, label="author", by="author")
enough = own_chunks.label.value_counts()
own_chunks = own_chunks[own_chunks.label.isin(enough[enough >= 3].index)].reset_index(drop=True)
print(f"{len(own_chunks)} chunks of {OWN_CHUNK} characters, "
      f"{own_chunks.label.nunique()} authors with three or more")

own_distance = clustering.fit(own_chunks.text.tolist())
own_real, own_shuffled = separation(own_distance, own_chunks.label.to_numpy())
print(f"silhouette by author: {own_real:.3f}   (shuffled labels: {own_shuffled:.3f})")

mine = PlotSettings(
    figsize=(7, 6),
    title=f"{OWN_CHUNK}-character chunks of your chat, by trigram counts",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(mine).plot(coordinates=clustering.reduce_dims(own_distance),
                                    labels=own_chunks.label.to_numpy(), palette="Set1")


## Waar deze baseline voor dient

Trigram-tellingen scheidden de 4chan-maanden omdat de claim gaat over *stijl* — interpunctie, spatiëring, woordkeuze — en sequenties van drie karakters dragen daar veel van mee. Je eigen chat zal gewoonlijk niet zo netjes splitsen, en dat is geen falen van de methode: een silhouette dichtbij het geschudde silhouette betekent dat deze mensen, in chunks van deze grootte, niet verschillend genoeg typen om uit elkaar gehaald te worden door karakters te tellen. Zeg dat dan ook, met de chunkgrootte en het aantal chunks per persoon in de zin.

Houd de vorm van de pipeline in gedachten — normaliseren, chunken, tellen, afstand, kijken — want [06.3](06.3-vectorspaces%20jeffrey.ipynb) en [06.4](06.4-your-own-vectors%20jeffrey.ipynb) doorlopen dezelfde stappen met één verandering: wat er geteld wordt, wordt niet langer door jou gekozen. En behoud de leesgewoonte uit §6.2.4: een plaatje dat zijn kader vult, een getal dat niet van het plaatje afhangt, en een blik op *welke* features het werk deden vóórdat een mechanisme wordt opgeschreven.